### Generate a summary for each product category using Generative AI

In [3]:
#import Libraries
import re, json, os
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

In [4]:
#Data Loading

DATA_PATH = "../data/processed/customer_reviews_product_clustered.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head(3))

Shape: (34614, 20)

Columns:
['product_name', 'asins', 'brand', 'categories', 'manufacturer', 'reviews.date', 'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating', 'reviews.title', 'reviews.text', 'clean_text', 'sentiment', 'text_length', 'asin_clean', 'valid_asin', 'product_name_clean', 'product_id', 'cluster', 'meta_category']


,product_name,asins,brand,categories,manufacturer,reviews.date,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.title,reviews.text,clean_text,sentiment,text_length,asin_clean,valid_asin,product_name_clean,product_id,cluster,meta_category
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Amazon,2017-01-13 00:00:00+00:00,yes,0.0,5,Kindle,This product so far has not disappointed. My c...,this product so far has not disappointed. my c...,positive,143,B01AHB9CN2,True,all new fire hd 8 tablet 8 hd display wi fi 16...,ASIN_B01AHB9CN2,3,Kindle & Fire Devices – Special Offers
1,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Amazon,2017-01-13 00:00:00+00:00,yes,0.0,5,very fast,great for beginner or experienced person. Boug...,great for beginner or experienced person. boug...,positive,75,B01AHB9CN2,True,all new fire hd 8 tablet 8 hd display wi fi 16...,ASIN_B01AHB9CN2,3,Kindle & Fire Devices – Special Offers
2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Amazon,2017-01-13 00:00:00+00:00,yes,0.0,5,Beginner tablet for our 9 year old son.,Inexpensive tablet for him to use and learn on...,inexpensive tablet for him to use and learn on...,positive,131,B01AHB9CN2,True,all new fire hd 8 tablet 8 hd display wi fi 16...,ASIN_B01AHB9CN2,3,Kindle & Fire Devices – Special Offers


In [5]:
#original categories
print("Original categories:")
print(df["meta_category"].value_counts(dropna=False))

print("\nNumber of categories:",
      df["meta_category"].nunique(dropna=True))

Original categories:
meta_category
Kindle & Fire Devices – Special Offers     18109
Echo & Smart Home Devices                  12318
Fire Kids Edition Tablets & Accessories     1696
Kindle Fire Tablets – 16GB                  1104
Echo & Kindle Fire Devices                   855
Amazon Device Accessories & Chargers         532
Name: count, dtype: int64

Number of categories: 6


In [6]:
#Save the original category permanently

# Keep the original category created by the clustering step.
df["category"] = df["meta_category"]

print(df["category"].value_counts())

category
Kindle & Fire Devices – Special Offers     18109
Echo & Smart Home Devices                  12318
Fire Kids Edition Tablets & Accessories     1696
Kindle Fire Tablets – 16GB                  1104
Echo & Kindle Fire Devices                   855
Amazon Device Accessories & Chargers         532
Name: count, dtype: int64


In [7]:
#Inspect product names before cleaning
print("Missing product names:",
      df["product_name_clean"].isna().sum())

print("\nExamples of product names:")
display(
    df["product_name_clean"]
    .dropna()
    .value_counts()
    .head(20)
)

Missing product names: 0

Examples of product names:


product_name_clean
fire tablet 7 display wi fi 8 gb includes special offers magenta                                                                                                                                           10962
unknown                                                                                                                                                                                                     6087
echo white echo white                                                                                                                                                                                       3976
amazon kindle paperwhite ebook reader 4 gb 6 monochrome paperwhite touchscreen wi fi black                                                                                                                  3176
all new fire hd 8 tablet 8 hd display wi fi 16 gb includes special offers magenta                                                                

In [8]:
#Clean duplicate product names

def dedupe_halves(s):
    """
    If a product name is accidentally repeated twice,
    keep only one copy.

    Example:
    A B C A B C
    becomes:
    A B C
    """
    
    words = s.split()
    half = len(words) // 2

    if (
        len(words) % 2 == 0
        and words[:half] == words[half:]
    ):
        return " ".join(words[:half])

    return s

In [9]:
#Clean product display names    
FIXES = {
    "Gb": "GB",
    "Hd": "HD",
    "Ppi": "ppi",
    "Tv": "TV",
    "Usb": "USB",
    "Oem": "OEM",
    "Wi Fi": "Wi-Fi",
    "6Th": "6th",
    "5Th": "5th",
    "4Th": "4th",
    "3Rd": "3rd",
    "2Nd": "2nd",
}


def display_name(s):
    """
    Makes a product name easier to read.

    This function changes presentation only.
    It does NOT decide which category the product belongs to.
    """

    if not isinstance(s, str):
        return s

    # Remove accidental duplicated half
    s = dedupe_halves(s)

    # Remove ASIN-like codes
    words = [
        w for w in s.split()
        if not re.fullmatch(r"b0[0-9a-z]{8}", w.lower())
    ]

    # Title case
    out = " ".join(words).title()

    # Restore important technical abbreviations
    for old, new in FIXES.items():
        out = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            out
        )

    # Formatting fixes
    out = re.sub(r"(\d+)Gb\b", r"\1 GB", out)
    out = re.sub(r"(\d+)In\b", r"\1-inch", out)

    return out

In [10]:
#Create the usable product label
name = df["product_name_clean"]

# Treat only genuinely missing/empty/unknown names as unusable.
unknown = (
    name.isna()
    | name.astype(str).str.strip().str.lower().isin(
        ["", "unknown", "missing", "nan"]
    )
)

df["product_label"] = name.where(~unknown, np.nan)

# Clean duplicated product names
df["product_label"] = df["product_label"].map(
    lambda x: dedupe_halves(x) if isinstance(x, str) else x
)

print("Reviews without usable product name:",
      df["product_label"].isna().sum())

print("Distinct usable product names:",
      df["product_label"].nunique())


Reviews without usable product name: 6087
Distinct usable product names: 47


In [11]:
#Check all six categories again
category_summary = (
    df.groupby("category")
      .agg(
          reviews=("reviews.rating", "size"),
          products=("product_label", "nunique")
      )
      .sort_values("reviews", ascending=False)
)

display(category_summary)

,reviews,products
category,,
Kindle & Fire Devices – Special Offers,18109,26
Echo & Smart Home Devices,12318,16
Fire Kids Edition Tablets & Accessories,1696,5
Kindle Fire Tablets – 16GB,1104,3
Echo & Kindle Fire Devices,855,3
Amazon Device Accessories & Chargers,532,3


In [12]:
#Inspect products inside each category

for category, group in df.groupby("category"):

    product_counts = (
        group["product_label"]
        .dropna()
        .value_counts()
    )

    print("\n" + "=" * 80)
    print(category)
    print("=" * 80)

    print("Total reviews:", len(group))
    print("Usable products:", len(product_counts))

    print("\nTop products:")
    
    for product, count in product_counts.head(10).items():
        print(f"{count:>6}  {display_name(product)}")


Amazon Device Accessories & Chargers
Total reviews: 532
Usable products: 3

Top products:
   211  All New Kindle E Reader Black 6 Glare Free Touchscreen Display Wi-Fi Includes Special Offers
   188  Amazon 5W USB Official OEM Charger And Power Adapter For Fire Tablets And Kindle Ereaders
     1  Amazon Kindle Fire HD 3rd Generation 8 GB

Echo & Kindle Fire Devices
Total reviews: 855
Usable products: 3

Top products:
   387  Echo White
   211  Fire Tablet 7 Display Wi-Fi 8 GB Includes Special Offers Magenta
     1  Amazon Kindle Fire HD 3rd Generation 8 GB

Echo & Smart Home Devices
Total reviews: 12318
Usable products: 16

Top products:
  3569  Echo White
  2527  Amazon Fire TV
   318  Amazon Amazon Tap Portable Bluetooth And Wi-Fi Speaker Black
   128  Amazon Fire HD 10 Tablet Wi-Fi 16 GB Special Offers Silver Aluminum
    36  Amazon 9W Powerfast Official OEM USB Charger And Power Adapter For Fire Tablets And Kindle Ereaders
     9  Kindle Dx Leather Cover Black Fits 9 7 Display Late

In [13]:
#Sentiment summary

def sentiment_summary(product_df):
    """
    Calculate the percentage of positive,
    neutral and negative reviews for one product.
    """

    counts = product_df["sentiment"].value_counts()
    total = len(product_df)

    if total == 0:
        return {
            "positive_pct": 0,
            "neutral_pct": 0,
            "negative_pct": 0,
        }

    return {
        "positive_pct": round(
            counts.get("positive", 0) / total * 100,
            1
        ),
        "neutral_pct": round(
            counts.get("neutral", 0) / total * 100,
            1
        ),
        "negative_pct": round(
            counts.get("negative", 0) / total * 100,
            1
        ),
    }

In [14]:
#Select review examples
def sample_reviews(rv, low, n, max_chars=280):
    """
    Select real review examples for Claude.

    low=True:
        select 1-3 star reviews

    low=False:
        select 4-5 star reviews

    We prefer reviews with more helpful votes.
    """

    if low:
        sub = rv[
            rv["reviews.rating"] <= 3
        ].copy()

        # 1-2 star reviews first, then 3 star
        sub["_severity"] = (
            sub["reviews.rating"] > 2
        ).astype(int)

    else:
        sub = rv[
            rv["reviews.rating"] >= 4
        ].copy()

        sub["_severity"] = 0

    # Remove extremely short/long reviews
    sub = sub[
        sub["reviews.text"]
        .fillna("")
        .str.len()
        .between(40, max_chars)
    ]

    # Sort severe reviews first,
    # then helpful reviews first
    sort_columns = ["_severity"]
    ascending = [True]

    if "reviews.numHelpful" in sub.columns:
        sort_columns.append("reviews.numHelpful")
        ascending.append(False)

    sub = (
        sub
        .sort_values(
            sort_columns,
            ascending=ascending
        )
        .head(n)
    )

    return [
        f"({int(rating)} stars) {text}"
        for rating, text in zip(
            sub["reviews.rating"],
            sub["reviews.text"]
        )
    ]

In [15]:
#Create product evidence
def product_facts(product_df, label, role):
    """
    Create the factual information Claude is allowed to use
    for one product.
    """

    return {
        "name": display_name(label),

        "role": role,

        "review_count": len(product_df),

        "average_rating": round(
            product_df["reviews.rating"].mean(),
            2
        ),

        **sentiment_summary(product_df),

        "negative_reviews_1_2_stars": int(
            (product_df["reviews.rating"] <= 2).sum()
        ),

        "reviews_3_stars_or_less": int(
            (product_df["reviews.rating"] <= 3).sum()
        ),

        "sample_of_positive_reviews": sample_reviews(
            product_df,
            low=False,
            n=4
        ),

        "sample_of_low_rated_reviews": sample_reviews(
            product_df,
            low=True,
            n=10
        ),
    }

In [16]:
#Build facts for ONE category
MIN_PRODUCT_REVIEWS = 10

GLOBAL_MEAN = df["reviews.rating"].mean()


def build_facts(df, category):

    # IMPORTANT:
    # Use the ORIGINAL category.
    cat = df[
        df["category"] == category
    ]

    # Only reviews with a usable product name
    named = cat[
        cat["product_label"].notna()
    ]

    # Count reviews for each product
    product_stats = (
        named
        .groupby("product_label")
        .agg(
            n_reviews=("reviews.rating", "size"),
            avg_rating=("reviews.rating", "mean")
        )
        .reset_index()
    )

    # Keep products with at least 10 reviews
    ranked = product_stats[
        product_stats["n_reviews"] >= MIN_PRODUCT_REVIEWS
    ].copy()

    # If no products meet the threshold,
    # return a category-level result instead of deleting the category.
    if len(ranked) == 0:

        return {
            "category": category,
            "reviews_in_category": len(cat),
            "products_ranked": 0,
            "min_reviews_used": MIN_PRODUCT_REVIEWS,
            "pct_reviews_without_product_name": round(
                (1 - len(named) / len(cat)) * 100,
                1
            ),
            "most_reviewed_product": None,
            "products": [],
            "data_warning": (
                "No product has enough reviews for reliable "
                "product-level ranking."
            ),
        }

    # Bayesian-style score.
    # This prevents a product with very few reviews
    # from automatically winning just because of a high rating.
    ranked["score"] = (
        (
            ranked["n_reviews"]
            /
            (ranked["n_reviews"] + MIN_PRODUCT_REVIEWS)
        )
        * ranked["avg_rating"]
        +
        (
            MIN_PRODUCT_REVIEWS
            /
            (ranked["n_reviews"] + MIN_PRODUCT_REVIEWS)
        )
        * GLOBAL_MEAN
    )

    ranked = ranked.sort_values(
        "score",
        ascending=False
    )

    # Most reviewed product
    best_seller = (
        ranked
        .sort_values(
            "n_reviews",
            ascending=False
        )
        .iloc[0]
    )

    # Take top 3, if available
    top_products = ranked.head(3)

    # Lowest-ranked product
    # Only mark as weakest if there is more than one product.
    if len(ranked) >= 2:
        weakest_product = ranked.iloc[-1]
    else:
        weakest_product = None

    products = []

    # Add top products
    for row in top_products.itertuples():

        role = "top"

        # If this is also the weakest product,
        # don't duplicate it later.
        if (
            weakest_product is not None
            and row.product_label
            == weakest_product["product_label"]
        ):
            role = "top_and_weakest"

        product_df = named[
            named["product_label"]
            == row.product_label
        ]

        products.append(
            product_facts(
                product_df,
                row.product_label,
                role
            )
        )

    # Add weakest product if it was not already included.
    if (
        weakest_product is not None
        and weakest_product["product_label"]
        not in top_products["product_label"].values
    ):

        product_df = named[
            named["product_label"]
            == weakest_product["product_label"]
        ]

        products.append(
            product_facts(
                product_df,
                weakest_product["product_label"],
                "weakest"
            )
        )

    return {
        "category": category,

        "reviews_in_category": len(cat),

        "products_ranked": len(ranked),

        "min_reviews_used": MIN_PRODUCT_REVIEWS,

        "pct_reviews_without_product_name": round(
            (1 - len(named) / len(cat)) * 100,
            1
        ),

        "most_reviewed_product": {
            "name": display_name(
                best_seller["product_label"]
            ),
            "review_count": int(
                best_seller["n_reviews"]
            ),
            "average_rating": round(
                float(best_seller["avg_rating"]),
                2
            ),
        },

        "products": products,

        "data_warning": (
            "Some products have relatively few reviews; "
            "rankings should be interpreted cautiously."
            if (ranked["n_reviews"] < 30).any()
            else None
        ),
    }

In [17]:
#Build facts for ALL SIX categories
categories = (
    df["category"]
    .dropna()
    .unique()
    .tolist()
)

print("Categories found:", len(categories))
print(categories)

Categories found: 6
['Kindle & Fire Devices – Special Offers', 'Fire Kids Edition Tablets & Accessories', 'Amazon Device Accessories & Chargers', 'Echo & Kindle Fire Devices', 'Kindle Fire Tablets – 16GB', 'Echo & Smart Home Devices']


In [18]:
all_facts = {}

for category in categories:

    facts = build_facts(
        df,
        category
    )

    all_facts[category] = facts

print("\nCategories prepared for Claude:",
      len(all_facts))

for category, facts in all_facts.items():

    print(
        f"- {category}: "
        f"{facts['reviews_in_category']} reviews, "
        f"{facts['products_ranked']} ranked products"
    )


Categories prepared for Claude: 6
- Kindle & Fire Devices – Special Offers: 18109 reviews, 14 ranked products
- Fire Kids Edition Tablets & Accessories: 1696 reviews, 2 ranked products
- Amazon Device Accessories & Chargers: 532 reviews, 2 ranked products
- Echo & Kindle Fire Devices: 855 reviews, 2 ranked products
- Kindle Fire Tablets – 16GB: 1104 reviews, 2 ranked products
- Echo & Smart Home Devices: 12318 reviews, 5 ranked products


In [19]:
#Specifically inspect Accessories & Chargers
ACCESSORIES_CATEGORY = "Amazon Device Accessories & Chargers"

facts = all_facts.get(
    ACCESSORIES_CATEGORY
)

print(
    json.dumps(
        facts,
        indent=2,
        ensure_ascii=False
    )
)

{
  "category": "Amazon Device Accessories & Chargers",
  "reviews_in_category": 532,
  "products_ranked": 2,
  "min_reviews_used": 10,
  "pct_reviews_without_product_name": 24.8,
  "most_reviewed_product": {
    "name": "All New Kindle E Reader Black 6 Glare Free Touchscreen Display Wi-Fi Includes Special Offers",
    "review_count": 211,
    "average_rating": 4.43
  },
  "products": [
    {
      "name": "Amazon 5W USB Official OEM Charger And Power Adapter For Fire Tablets And Kindle Ereaders",
      "role": "top",
      "review_count": 188,
      "average_rating": 4.45,
      "positive_pct": 86.7,
      "neutral_pct": 2.7,
      "negative_pct": 10.6,
      "negative_reviews_1_2_stars": 20,
      "reviews_3_stars_or_less": 25,
      "sample_of_positive_reviews": [
        "(5 stars) Bought to use with Kindle. Works fine! Good quality and charges the kindle quickly.",
        "(5 stars) Works just as you'd expect. Small and compact which is nice, and charges the kindle really quickly

In [20]:
#Check every category before Claude

for category, facts in all_facts.items():

    print("\n" + "=" * 80)
    print(category)
    print("=" * 80)

    print("Reviews:",
          facts["reviews_in_category"])

    print("Ranked products:",
          facts["products_ranked"])

    print("Missing product name:",
          facts["pct_reviews_without_product_name"],
          "%")

    if facts["most_reviewed_product"]:
        print(
            "Most reviewed:",
            facts["most_reviewed_product"]["name"]
        )

    print("\nProducts:")

    for product in facts["products"]:
        print(
            f"  - {product['name']}"
            f" | {product['average_rating']} rating"
            f" | {product['review_count']} reviews"
            f" | {product['role']}"
        )

    if facts.get("data_warning"):
        print(
            "\nWARNING:",
            facts["data_warning"]
        )


Kindle & Fire Devices – Special Offers
Reviews: 18109
Ranked products: 14
Missing product name: 0.0 %
Most reviewed: Fire Tablet 7 Display Wi-Fi 8 GB Includes Special Offers Magenta

Products:
  - Amazon Kindle Paperwhite Ebook Reader 4 GB 6 Monochrome Paperwhite Touchscreen Wi-Fi Black | 4.77 rating | 2928 reviews | top
  - Fire HD 8 Tablet With Alexa 8 HD Display 32 GB Tangerine With Special Offers | 4.86 rating | 14 reviews | top
  - Kindle Voyage E Reader 6 High Resolution Display 300 ppi With Adaptive Built In Light Pagepress Sensors Wi-Fi Includes Special Offers | 4.74 rating | 580 reviews | top
  - Fire Tablet 7 Display Wi-Fi 8 GB Includes Special Offers Magenta | 4.45 rating | 10751 reviews | weakest


Fire Kids Edition Tablets & Accessories
Reviews: 1696
Ranked products: 2
Missing product name: 0.0 %
Most reviewed: Fire Kids Edition Tablet 7 Display Wi-Fi 16 GB Green Kid Proof Case

Products:
  - Amazon Kindle Paperwhite Ebook Reader 4 GB 6 Monochrome Paperwhite Touchscreen W

In [21]:
#Save the structured evidence
REPORTS_DIR = Path("../reports")
SUMMARY_DATA_DIR = REPORTS_DIR / "summaries_data"

REPORTS_DIR.mkdir(
    exist_ok=True
)

SUMMARY_DATA_DIR.mkdir(
    exist_ok=True
)

for category, facts in all_facts.items():

    slug = re.sub(
        r"[^a-z0-9]+",
        "_",
        category.lower()
    ).strip("_")

    output_file = (
        SUMMARY_DATA_DIR
        / f"{slug}.json"
    )

    output_file.write_text(
        json.dumps(
            facts,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

print(
    f"Saved evidence for {len(all_facts)} categories."
)

Saved evidence for 6 categories.


## Claude API Integration

In [37]:
#Connect to Claude
import anthropic
from dotenv import load_dotenv

load_dotenv()

api_key_found = bool(
    os.getenv("ANTHROPIC_API_KEY")
)

print(
    "API key found:",
    api_key_found
)

client = anthropic.Anthropic()

MODEL = "claude-haiku-4-5"

API key found: True


In [38]:
#Claude system prompt
SYSTEM_STRICT = """
You are a factual consumer product review analyst.

Your job is to create a short buying guide using ONLY the supplied evidence.

IMPORTANT RULES:

1. Never invent facts, specifications, prices, features or products.

2. Every rating, percentage and review count must come from the supplied data.

3. Product specifications may only be mentioned when they appear
   directly in the product name or review samples.

4. Use the positive review samples to describe praise.

5. Use the low-rated review samples to identify complaint themes.

6. When discussing complaint themes, clearly say they come from
   the sampled low-rated reviews.
   Do NOT treat the sample as the complete set of complaints.

7. Never say that a complaint appears in X% of all reviews
   unless that exact number is provided in the data.

8. Low-rated samples may contain praise or irrelevant text.
   Ignore those when identifying complaints.

9. Compare products against each other.
   A "weakest" product means lowest-ranked in this category,
   not necessarily a bad product.

10. If products have similar ratings, say that the difference is small.

11. If the evidence is thin, say so honestly.

12. Do not move products between categories.
    The supplied category is authoritative.

13. Use the product names exactly as supplied.

14. Do not claim that reviews were verified.

15. Keep recommendations consistent with the evidence.
"""

In [39]:
#Claude output format
FORMAT_STRICT = """
Write a consumer buying guide in Markdown.

Use exactly these sections:

# {category} Buying Guide

## Overview

Explain:
- what this category contains
- how many reviews are included
- how many products were ranked

Mention if the category has limited evidence.

## Top Products

Cover the top products supplied in the data.

For each product include:
- average rating
- positive percentage
- what buyers praise
- how it differs from the other products

If there are only 2 products, discuss both.
Do NOT invent a third product.

## Main Complaints

For each relevant product:
- identify the main complaint themes
- give sample counts such as "4 of the 10 sampled low-rated reviews"
- explain how common serious complaints are overall using the supplied data

Only use short direct quotes when useful.
A quote must be copied exactly from the supplied review samples
and must be no more than 12 words.

## Weakest Product

Identify the lowest-ranked product when one exists.

Explain:
- its rating
- its negative-review percentage
- its main complaint themes

If the difference is small, explicitly say that it is only relatively weaker.

## Recommendation

Give a short recommendation explaining which product
may suit which type of buyer.

Do not invent buyer characteristics that are not supported
by the supplied evidence.

Keep the guide around 300-380 words.
"""


In [41]:
#Function to send ONE category to Claude
def generate_guide(facts):

    prompt = (
        "CATEGORY EVIDENCE:\n"
        + json.dumps(
            facts,
            indent=2,
            ensure_ascii=False
        )
        + "\n\n"
        + FORMAT_STRICT.format(
            category=facts["category"]
        )
    )

    message = client.messages.create(
        model=MODEL,
        max_tokens=1400,
        system=SYSTEM_STRICT,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    if message.stop_reason == "max_tokens":
        print(
            "WARNING: Claude stopped because "
            "the maximum token limit was reached."
        )

    return message.content[0].text

In [ ]:
#Test Accessories & Chargers FIRST
test_facts = all_facts[
    "Amazon Device Accessories & Chargers"
]

test_article = generate_guide(
    test_facts
)

print(test_article)

In [42]:
#Generate all six Categories
all_summaries = {}

for category, facts in all_facts.items():

    print("\n" + "=" * 80)
    print(category)
    print("=" * 80)

    article = generate_guide(
        facts
    )

    all_summaries[category] = article

    print(article)


Kindle & Fire Devices – Special Offers
# Kindle & Fire Devices – Special Offers Buying Guide

## Overview

This category includes Kindle e-readers and Fire tablets sold with Amazon Special Offers. The guide covers 14 ranked products based on 18,109 customer reviews. Note that some products have relatively few reviews; rankings should be interpreted cautiously.

## Top Products

**Amazon Kindle Paperwhite Ebook Reader 4 GB 6 Monochrome Paperwhite Touchscreen Wi-Fi Black**
- Rating: 4.77 / 5.0 | 97.3% positive
- Buyers praise the backlit display, noting "back lighting is great" and appreciating variable lighting for reading at night. The touchscreen setup is easy, and readers highlight that it feels "like reading a piece of paper."

**Kindle Voyage E Reader 6 High Resolution Display 300 ppi With Adaptive Built In Light Pagepress Sensors Wi-Fi Includes Special Offers**
- Rating: 4.74 / 5.0 | 96.9% positive
- This model earns praise for its high-resolution display described as "one hundre

## Generate Summeries

In [43]:
#Save the six Markdown guides
SUMMARY_DIR = REPORTS_DIR / "summaries"

SUMMARY_DIR.mkdir(
    exist_ok=True
)

for category, article in all_summaries.items():

    slug = re.sub(
        r"[^a-z0-9]+",
        "_",
        category.lower()
    ).strip("_")

    output_file = (
        SUMMARY_DIR
        / f"{slug}.md"
    )

    output_file.write_text(
        f"# {category}\n\n{article}\n",
        encoding="utf-8"
    )

print(
    f"Saved {len(all_summaries)} category guides."
)

Saved 6 category guides.


In [22]:
# Create category summary/index JSON for dashboard
summary_index = []


for category, facts in all_facts.items():

    slug = re.sub(
        r"[^a-z0-9]+",
        "_",
        category.lower()
    ).strip("_")


    summary_index.append(
        {
            "category": category,
            "reviews_in_category": facts["reviews_in_category"],
            "products_ranked": facts["products_ranked"],
            "pct_reviews_without_product_name": (
                facts["pct_reviews_without_product_name"]
            ),
            "most_reviewed_product": (
                facts["most_reviewed_product"]["name"]
                if facts["most_reviewed_product"]
                else None
            ),
            "detail_file": f"{slug}.json"
        }
    )


# --------------------------------------------------
# Save category summary
# --------------------------------------------------

CATEGORY_SUMMARY_PATH = (
    SUMMARY_DATA_DIR / "category_summary.json"
)


CATEGORY_SUMMARY_PATH.write_text(
    json.dumps(
        summary_index,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    f"Saved category summary to: "
    f"{CATEGORY_SUMMARY_PATH}"
)

Saved category summary to: ../reports/summaries_data/category_summary.json
